# VADER vs Transformer: Does the Sentiment Method Change the Conclusion?

Many studies use **VADER**, a word-list sentiment tool, to score text and then draw conclusions from those scores.
This notebook re-scores the same texts with a **Transformer** model (`cardiffnlp/twitter-roberta-base-sentiment-latest`)
and checks which conclusions still hold when the measurement method changes.

The example data comes from a study of the first 30 seconds of educational YouTube videos.
The data is not included; Step 1 lists the columns you need, so you can run the same check on your own data.

**How to run:** open in Google Colab → `Runtime → Change runtime type → T4 GPU` → `Runtime → Run all`.

## Step 0 · Install

In [ ]:
!pip -q install vaderSentiment
import torch
print("GPU available:", torch.cuda.is_available())

## Step 1 · Load and merge the data

Expected files:

| File | Columns used |
|---|---|
| `full_size_video_metrics.csv` | `id`, `channel_name`, `views`, `likes`, `video_duration` (ISO 8601, e.g. `PT15M7S`), `sentiment_score` (VADER compound score from the original analysis), `video_rank_category` |
| `transcripts_combined.csv` | `id`, `transcript` (first 30 seconds, English) |

Videos with an empty transcript are dropped.

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose BOTH csv files

In [ ]:
import re
import numpy as np
import pandas as pd

metrics = pd.read_csv("full_size_video_metrics.csv")
transcripts = pd.read_csv("transcripts_combined.csv", keep_default_na=False)

df = metrics.merge(transcripts, on="id", how="left")
df["transcript"] = df["transcript"].fillna("").str.strip()
df = df[df["transcript"] != ""].copy()
print("videos with a transcript:", len(df))

## Step 2 · Recompute VADER

VADER is recomputed on the same text and compared with the original scores, to confirm both methods score the same text.
The analysis then uses the original scores, so it tests the exact numbers the original conclusions were based on.

Labels use the standard VADER thresholds: compound ≥ 0.05 → positive, ≤ −0.05 → negative, otherwise neutral.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()
df["vader_score"] = df["transcript"].apply(lambda t: vader.polarity_scores(t)["compound"])

In [ ]:
diff = (df["vader_score"] - df["sentiment_score"]).abs()
print("correlation with the original scores:", round(df["vader_score"].corr(df["sentiment_score"]), 4))
print("share of videos that differ by more than 0.01:", round((diff > 0.01).mean(), 4))

def vader_label(x):
    if x >= 0.05:
        return "positive"
    if x <= -0.05:
        return "negative"
    return "neutral"

# Use the original scores as "VADER", so we test the exact numbers the original conclusions used.
df["vader_label"] = df["sentiment_score"].apply(vader_label)
df["vader_label"].value_counts(normalize=True).round(4)

## Step 3 · Transformer scoring

The model outputs three probabilities for each transcript: negative, neutral, positive.

- `rb_label`: the class with the highest probability
- `rb_score`: P(positive) − P(negative), from −1 to 1, comparable to VADER's compound score

Inputs are truncated at 512 tokens; 30 seconds of speech is well below that.

In [ ]:
from transformers import pipeline

MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
clf = pipeline("text-classification", model=MODEL, top_k=None,
               device=0 if torch.cuda.is_available() else -1)

outputs = clf(df["transcript"].tolist(), batch_size=64, truncation=True, max_length=512)

def to_probs(scores):
    p = {s["label"].lower(): s["score"] for s in scores}
    return p["negative"], p["neutral"], p["positive"]

probs = np.array([to_probs(o) for o in outputs])
df["rb_neg"], df["rb_neu"], df["rb_pos"] = probs[:, 0], probs[:, 1], probs[:, 2]

In [ ]:
df["rb_score"] = df["rb_pos"] - df["rb_neg"]
df["rb_label"] = df[["rb_neg", "rb_neu", "rb_pos"]].values.argmax(axis=1)
df["rb_label"] = df["rb_label"].map({0: "negative", 1: "neutral", 2: "positive"})
df["rb_label"].value_counts(normalize=True).round(4)

## Step 4 · Do the two methods agree?

- **Confusion table**: rows = VADER, columns = Transformer
- **Agreement**: share of videos with the same label
- **Cohen's kappa**: agreement after removing chance agreement (0 = chance level, 1 = perfect)
- **Spearman correlation**: do the two continuous scores rank videos in the same order?

In [ ]:
from scipy import stats
from sklearn.metrics import cohen_kappa_score

order = ["negative", "neutral", "positive"]
print(pd.crosstab(df["vader_label"], df["rb_label"], margins=True).reindex(index=order + ["All"], columns=order + ["All"]))
print()
print("agreement:", round((df["vader_label"] == df["rb_label"]).mean(), 4))
print("Cohen's kappa:", round(cohen_kappa_score(df["vader_label"], df["rb_label"]), 4))
rho, p = stats.spearmanr(df["sentiment_score"], df["rb_score"])
print("Spearman rho:", round(rho, 4), " p =", f"{p:.2g}")

### Where do they disagree?

Reading a few disagreements shows *why* a word list and a context-aware model see the same text differently.

In [ ]:
pd.set_option("display.max_colwidth", 300)
cols = ["channel_name", "sentiment_score", "rb_score", "transcript"]
print("VADER positive, Transformer negative")
display(df[(df.vader_label == "positive") & (df.rb_label == "negative")].sample(5, random_state=0)[cols])
print("VADER negative, Transformer positive")
display(df[(df.vader_label == "negative") & (df.rb_label == "positive")].sample(5, random_state=0)[cols])

## Step 5 · Define the tiers

`video_rank_category` splits each channel's videos into **top_videos** (the more-viewed group) and
**bottom_videos** (the less-viewed group). These are used as **hit** vs **regular**.

Channels labelled `unknown` (TED in this dataset) are split at the channel's median view count instead.

Note: this is a coarser split than the hit levels used in the original study, so absolute numbers are not comparable with it.
The comparison that matters is VADER vs Transformer under the same split.

In [ ]:
def tier(row):
    cat = row["video_rank_category"]
    if isinstance(cat, str) and cat.endswith("top_videos"):
        return "top"
    if isinstance(cat, str) and cat.endswith("bottom_videos"):
        return "bottom"
    return None

df["tier"] = df.apply(tier, axis=1)
ted = df["channel_name"] == "TED"
ted_median = df.loc[ted, "views"].median()
df.loc[ted, "tier"] = np.where(df.loc[ted, "views"] >= ted_median, "top", "bottom")

print(df["tier"].value_counts(dropna=False))

## Step 6 · Re-test each finding

Each analysis runs twice: once with VADER labels and once with Transformer labels.

### Finding 1 · Negative openings are more common in hit videos

Test: chi-square test of tier × sentiment label.

In [ ]:
results = []   # collects one row per check for the final summary

def share_table(data, label_col):
    t = pd.crosstab(data["tier"], data[label_col], normalize="index").reindex(columns=order) * 100
    return t.round(2)

def c1(data, label_col, name):
    t = share_table(data, label_col)
    counts = pd.crosstab(data["tier"], data[label_col])
    chi2, p, _, _ = stats.chi2_contingency(counts)
    neg_top, neg_bottom = t.loc["top", "negative"], t.loc["bottom", "negative"]
    holds = bool((neg_top > neg_bottom) and p < 0.05)
    print(f"--- {name} ({label_col}) ---")
    print(t)
    print(f"negative: top {neg_top}% vs bottom {neg_bottom}%  (ratio {neg_top / neg_bottom:.2f}),  chi-square p = {p:.3g}")
    return holds, f"top {neg_top}% vs bottom {neg_bottom}%, p={p:.3g}"

d = df[df["tier"].notna()]
for lab, meth in [("vader_label", "VADER"), ("rb_label", "Transformer")]:
    holds, detail = c1(d, lab, "All channels")
    results.append({"conclusion": "1. More negative openings in hits", "method": meth, "holds": holds, "detail": detail})
    print()

### Finding 2 · Negative openings go with longer videos

The original study reported this for individual channels. Here it is tested across all hits and within two example channels.
Direction is judged by the mean (as in the original study); significance by a Mann–Whitney test (negative vs positive).

In [ ]:
def iso_seconds(s):
    # 'PT1H2M3S' -> 3723 seconds
    if not isinstance(s, str):
        return np.nan
    m = re.fullmatch(r"P(?:(\d+)D)?T?(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", s)
    if not m:
        return np.nan
    d, h, mi, se = (int(x) if x else 0 for x in m.groups())
    return d * 86400 + h * 3600 + mi * 60 + se

df["duration_s"] = df["video_duration"].apply(iso_seconds)

def c2(data, label_col, name):
    g = data.groupby(label_col)["duration_s"].agg(["count", "mean", "median"]).reindex(order).round(0)
    neg = data.loc[data[label_col] == "negative", "duration_s"].dropna()
    pos = data.loc[data[label_col] == "positive", "duration_s"].dropna()
    p = stats.mannwhitneyu(neg, pos).pvalue if len(neg) > 2 and len(pos) > 2 else np.nan
    print(f"--- {name} ({label_col}) ---")
    print(g)
    print(f"Mann-Whitney negative vs positive: p = {p:.3g}")
    # direction by mean (as in the original study), significance by rank test
    holds = None if np.isnan(p) else bool(g.loc["negative", "mean"] > g.loc["positive", "mean"] and p < 0.05)
    return holds, f"mean neg {g.loc['negative', 'mean']:.0f}s vs pos {g.loc['positive', 'mean']:.0f}s, p={p:.3g}"

for scope, data in [("All hits", df[df.tier == "top"]),
                    ("Vsauce hits", df[(df.tier == "top") & (df.channel_name == "Vsauce")]),
                    ("Doctor Mike hits", df[(df.tier == "top") & (df.channel_name == "Doctor Mike")])]:
    for lab, meth in [("vader_label", "VADER"), ("rb_label", "Transformer")]:
        holds, detail = c2(data, lab, scope)
        results.append({"conclusion": f"2. Negative = longer ({scope})", "method": meth, "holds": holds, "detail": detail})
        print()

### Finding 3 · The "watch but don't like" pattern

Original finding: negative openings have the lowest like/view ratio. Test: Kruskal–Wallis across the three sentiment groups.

In [ ]:
df["like_rate"] = np.where(df["views"] > 0, df["likes"] / df["views"] * 100, np.nan)

def c3(data, label_col, name):
    data = data.dropna(subset=["like_rate"])
    g = data.groupby(label_col)["like_rate"].agg(["count", "mean", "median"]).reindex(order).round(3)
    groups = [data.loc[data[label_col] == k, "like_rate"] for k in order]
    p = stats.kruskal(*groups).pvalue if all(len(x) > 2 for x in groups) else np.nan
    print(f"--- {name} ({label_col}) ---")
    print(g)
    print(f"Kruskal-Wallis p = {p:.3g}")
    lowest = g["mean"].idxmin()
    holds = None if np.isnan(p) else bool(lowest == "negative" and p < 0.05)
    return holds, f"lowest mean: {lowest}, p={p:.3g}"

for scope, data in [("All hits", df[df.tier == "top"]),
                    ("Doctor Mike hits", df[(df.tier == "top") & (df.channel_name == "Doctor Mike")])]:
    for lab, meth in [("vader_label", "VADER"), ("rb_label", "Transformer")]:
        holds, detail = c3(data, lab, scope)
        results.append({"conclusion": f"3. Negative = lowest like rate ({scope})", "method": meth, "holds": holds, "detail": detail})
        print()

### Finding 4 · Channel differences

Original finding: one science channel's biggest hits mostly open positively, and TED does not follow the "negative → hit" pattern.

In [ ]:
def c4(data, label_col, name):
    t = share_table(data, label_col)
    print(f"--- {name} ({label_col}) ---")
    print(t)
    return t

for lab, meth in [("vader_label", "VADER"), ("rb_label", "Transformer")]:
    t = c4(df[df.channel_name == "Vsauce"], lab, "Vsauce")
    holds = bool(t.loc["top", "positive"] > 50)
    results.append({"conclusion": "4a. Vsauce hits mostly positive", "method": meth, "holds": holds,
                    "detail": f"top positive {t.loc['top', 'positive']}%"})
    t = c4(df[df.channel_name == "TED"], lab, "TED")
    holds = bool(t.loc["top", "negative"] < t.loc["bottom", "negative"])
    results.append({"conclusion": "4b. TED: fewer negative openings in hits", "method": meth, "holds": holds,
                    "detail": f"top {t.loc['top', 'negative']}% vs bottom {t.loc['bottom', 'negative']}%"})
    print()

### Finding 5 · Sentiment vs creator persona

Persona labels are not in this dataset, so this finding is not re-tested here.

## Step 7 · Summary

✅ = the finding holds with this method (right direction and, where a test is used, p < 0.05). ❌ = it does not.
⚠️ = too few videos in one group to run the test.

In [ ]:
summary = pd.DataFrame(results)
summary["holds"] = summary["holds"].map({True: "✅", False: "❌"}).fillna("⚠️")
table = summary.pivot(index="conclusion", columns="method", values="holds")[["VADER", "Transformer"]]
display(table)
display(summary)

## Step 8 · Save the results

Downloads the per-video scores and the summary table. **Do not commit these files to a public repository.**

In [ ]:
out_cols = ["id", "channel_name", "title", "views", "likes", "duration_s", "tier", "video_rank_category",
            "sentiment_score", "vader_label", "rb_neg", "rb_neu", "rb_pos", "rb_score", "rb_label"]
df[out_cols].to_csv("vader_vs_transformer_scores.csv", index=False)
summary.to_csv("conclusion_check_summary.csv", index=False)
files.download("vader_vs_transformer_scores.csv")
files.download("conclusion_check_summary.csv")